# scitex.stats -- Publication-Ready Statistics

In [1]:
import scitex as stx
import numpy as np

np.random.seed(42)

## Run a Statistical Test

In [2]:
# Two sample groups
g1 = np.random.normal(loc=5.0, scale=1.0, size=30)
g2 = np.random.normal(loc=6.5, scale=1.2, size=30)

# Run an independent t-test; result returned as a dict (default)
result = stx.stats.run_test("ttest_ind", g1, g2)
for key, value in result.items():
    print(f"  {key}: {value}")

  test_method: Student's t-test (independent)
  statistic: -5.8896512231854805
  stat_symbol: t
  alternative: two-sided
  n_x: 30
  n_y: 30
  var_x: x
  var_y: y
  pvalue: 2.0707465153913663e-07
  stars: ***
  alpha: 0.05
  significant: True
  effect_size: -1.5207014068245004
  effect_size_metric: Cohen's d
  effect_size_interpretation: large
  power: 0.9999359083688276
  H0: μ(x) = μ(y)
  p_value: 2.0707465153913663e-07
  test: Student's t-test (independent)
  formatted: t = -5.890, p = 0.0000, Cohen's d = -1.521, ***


## Get Test Recommendations

In [3]:
# recommend_tests requires a StatContext describing the experimental design
ctx = stx.stats.StatContext(
    n_groups=2,
    sample_sizes=[len(g1), len(g2)],
    outcome_type="continuous",
    design="between",
    paired=False,
    has_control_group=False,
    n_factors=1,
)
recommendations = stx.stats.recommend_tests(ctx, top_k=5)
print("Recommended tests:")
for t in recommendations:
    print(f"  {t}")

Recommended tests:
  brunner_munzel
  ttest_ind
  mannwhitneyu


## Effect Size

In [4]:
# Cohen's d effect size
# effect_sizes is a module; use effect_sizes.cohens_d directly
es = stx.stats.effect_sizes.cohens_d(g1, g2)
interpretation = stx.stats.effect_sizes.interpret_cohens_d(es)
print(f"Cohen's d = {es:.4f} ({interpretation})")

Cohen's d = -1.5207 (large)


## Multiple Comparison Correction

In [5]:
# Bonferroni correction for multiple comparisons
# correct_bonferroni takes a list of dicts (or DataFrame) with a 'pvalue' key
pvalue_list = [
    {"pvalue": 0.01},
    {"pvalue": 0.04},
    {"pvalue": 0.06},
]
corrected = stx.stats.correct.correct_bonferroni(pvalue_list, verbose=False)
for orig, adj in zip(pvalue_list, corrected):
    print(f"  p = {orig['pvalue']:.2f}  ->  p_adj = {adj['pvalue_adjusted']:.4f}  {adj['pstars']}")

  p = 0.01  ->  p_adj = 0.0300  *
  p = 0.04  ->  p_adj = 0.1200  ns
  p = 0.06  ->  p_adj = 0.1800  ns


## Format for Publication

In [6]:
# Format the test result for publication
# result is a dict from run_test; use p_to_stars and format_test_line
pvalue = result["pvalue"]
stars = stx.stats.p_to_stars(pvalue)
formatted = stx.stats.auto.format_test_line(result)
print(f"p = {pvalue:.4f} {stars}")
print(f"Formatted: {formatted}")

p = 0.0000 ***
Formatted: 


## Available Tests

In [7]:
# List all 23+ built-in statistical tests
tests = stx.stats.available_tests()
for t in tests:
    print(f"  {t}")

  anova


  brunner_munzel
  brunnermunzel
  chi2
  fisher
  friedman
  kendall
  kruskal
  ks_1samp
  ks_2samp
  mann_whitney
  mannwhitneyu
  pearson
  shapiro
  spearman
  ttest
  ttest_1samp
  ttest_ind
  ttest_paired
  ttest_rel
  wilcoxon


## Summary

`stx.stats` provides a unified interface for common statistical workflows:

- **`run_test()`** -- 23+ tests (t-test, Mann-Whitney, ANOVA, chi-squared, etc.)
- **`recommend_tests()`** -- automatic test selection based on data properties
- **`effect_sizes.cohens_d()`** -- Cohen's d; also Cliff's delta, eta-squared, etc.
- **`correct.correct_bonferroni()`** -- Bonferroni correction; also FDR, Holm, Sidak
- **`p_to_stars()`** / **`auto.format_test_line()`** -- publication-ready formatting

All `run_test()` calls accept `return_as="dict"` (default) or `return_as="dataframe"` for flexible downstream use.